# Construire une solution sur `2_medium`
Même démarche que pour `1_example` : **qui peut remplir quel besoin ? qu'est-ce que ça coûte ? quels choix s'excluent ?** Puis on écrit des plans et on les fait vérifier par `test_solution.py`.

Le changement : 3 000 modèles et 10 000 datasets. On ne peut plus « tout essayer » comme dans le notebook 08. On remplace l'énumération par des **règles de choix** (heuristiques gloutonnes), et on mesure leur qualité grâce à une **borne supérieure** calculée exactement.

**Prérequis** : `datasets/2_medium.json` et `test_solution.py` (à la racine du projet), `pandas`, `numpy`, `scipy`.

## 1. Charger l'instance

In [1]:
from pathlib import Path
from math import comb
import sys, json, time, bisect
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.sparse import csr_matrix, hstack, vstack, identity
pd.set_option("display.width", 150)

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from test_solution import getSolutionScore

DATASET = ROOT_DIR / "datasets" / "2_medium.json"
DATASET_TXT = DATASET.read_text()
instance = json.loads(DATASET_TXT)
energy_cap = instance["energyCap"]
datasets = pd.DataFrame(instance["datasets"]).rename(columns={"id": "dataset_id", "dataType": "data_type", "isCopyrighted": "is_copyrighted"})
models = pd.DataFrame(instance["models"])[["id", "value", "energyCost"]].rename(columns={"id": "model_id", "energyCost": "energy_cost"})
requirements = pd.DataFrame([(m["id"], t, b["lowerBound"], b["upperBound"]) for m in instance["models"]
                             for t, b in m["dataRequirements"].items()], columns=["model_id", "data_type", "lower", "upper"])
models["n_req"] = models.model_id.map(requirements.groupby("model_id").size())
models = models.set_index("model_id")
print("plafond d'énergie :", energy_cap)
print(f"{len(models)} modèles, {len(datasets)} datasets, {len(requirements)} besoins")
display(models.head()); display(datasets.head()); display(requirements.head())

plafond d'énergie : 209400189201537
3000 modèles, 10000 datasets, 6052 besoins


,value,energy_cost,n_req
model_id,,,
0,336,374735884465,1
1,746,808434371752,1
2,510,637464766112,1
3,488,578396905301,1
4,171,306283952002,2


,dataset_id,data_type,size,is_copyrighted
0,0,c,3690,1
1,1,n,1529,0
2,2,t,9127,1
3,3,t,6434,1
4,4,n,7360,1


,model_id,data_type,lower,upper
0,0,c,27433,30916
1,1,n,79024,81915
2,2,i,42550,44603
3,3,i,36907,39632
4,4,n,5756,9044


## 2. Qui peut remplir quel besoin ?
Dans le notebook 08, on énumérait toutes les combinaisons de datasets. Ici, chaque type compte environ 2 500 datasets et un besoin demande en moyenne 3 à 4 datasets. Le nombre de combinaisons explose :

In [2]:
for k in range(1, 6):
    print(f"combinaisons de {k} datasets parmi 2 500 : {comb(2500, k):.2e}")

combinaisons de 1 datasets parmi 2 500 : 2.50e+03
combinaisons de 2 datasets parmi 2 500 : 3.12e+06
combinaisons de 3 datasets parmi 2 500 : 2.60e+09
combinaisons de 4 datasets parmi 2 500 : 1.62e+12
combinaisons de 5 datasets parmi 2 500 : 8.11e+14


Impossible de tout essayer. On change de question : **au lieu de « quelles combinaisons ? », on demande « y a-t-il assez de données, et de quelle sorte ? »**

### 2a. L'offre et la demande, par type

In [3]:
requirements["impossible"] = requirements.upper < requirements.lower          # upper < lower : aucun dataset ne peut convenir
valid_req = requirements[~requirements.impossible]

bilan = pd.DataFrame({
    "offre_totale": datasets.groupby("data_type")["size"].sum(),
    "offre_libre": datasets[datasets.is_copyrighted == 0].groupby("data_type")["size"].sum(),
    "demande_min_totale": valid_req.groupby("data_type").lower.sum(),
    "besoin_médian": valid_req.groupby("data_type").lower.median(),
    "dataset_médian": datasets.groupby("data_type")["size"].median(),
})
bilan["besoins_servis_max"] = (bilan.offre_totale / bilan.besoin_médian).astype(int)
bilan["besoins_servis_max_libres"] = (bilan.offre_libre / bilan.besoin_médian).astype(int)
bilan

,offre_totale,offre_libre,demande_min_totale,besoin_médian,dataset_médian,besoins_servis_max,besoins_servis_max_libres
data_type,,,,,,,
c,12206646,4091830,35358162,17916.5,4940.0,681,228
i,12666573,4433636,35029963,18364.0,5011.0,689,241
n,12195484,3942452,35189208,17969.5,4826.0,678,219
t,12374546,4180286,37400806,18680.0,4731.0,662,223


**Lecture**
- Un besoin médian fait environ 18 000 unités et un dataset médian environ 5 000 : il faut **3 à 4 datasets par besoin**.
- Chaque type peut servir environ 700 besoins au total, mais seulement **220 à 240 avec des données libres**. Les données libres sont la ressource rare.

### 2b. Les besoins impossibles
Comme dans le notebook 08 avec le modèle 1, certains besoins ne peuvent pas être remplis par des datasets. Ici c'est encore plus net : `upper < lower`. Seule une **source** peut les couvrir.

In [4]:
impossible_models = set(requirements[requirements.impossible].model_id)
print("besoins impossibles :", requirements.impossible.sum(), "| modèles concernés :", len(impossible_models))
display(requirements[requirements.impossible].head())
print("valeur de ces modèles :", models.loc[sorted(impossible_models), "value"].describe()[["min", "50%", "max"]].to_dict())
print("énergie de ces modèles :", models.loc[sorted(impossible_models), "energy_cost"].describe()[["min", "50%", "max"]].to_dict())

besoins impossibles : 81 | modèles concernés : 58


,model_id,data_type,lower,upper,impossible
90,44,i,1,-537,True
91,44,n,1,-1752,True
141,68,t,1,-1439,True
216,106,n,1,-1203,True
612,311,t,1,-851,True


valeur de ces modèles : {'min': 1.0, '50%': 38.0, 'max': 96.0}
énergie de ces modèles : {'min': 1.0, '50%': 69112524841.0, 'max': 293866636741.0}


### 2c. Remplir un besoin sans tout énumérer
On garde, pour chaque type et chaque statut (libre / sous droits), les datasets **triés par taille**. Pour remplir `[lower, upper]` :
1. s'il existe un dataset qui termine le remplissage (taille dans `[lower − total, upper − total]`), on prend **le plus petit** : on gaspille le moins possible ;
2. sinon on prend **le plus gros** qui ne fait pas dépasser `upper`, puis on recommence.

Grâce au tri, chaque recherche est une recherche dichotomique (`bisect`) : c'est très rapide.

In [5]:
SIZE = dict(zip(datasets.dataset_id, datasets["size"]))
TYPE = dict(zip(datasets.dataset_id, datasets.data_type))
COPY = dict(zip(datasets.dataset_id, datasets.is_copyrighted))
NEEDS = {m: list(g[["data_type", "lower", "upper"]].itertuples(index=False)) for m, g in valid_req.groupby("model_id")}

class Pool:
    """Réserve de datasets d'un type et d'un statut, triée par taille."""
    def __init__(self, df):
        df = df.sort_values("size")
        self.sizes, self.ids = df["size"].tolist(), df.dataset_id.tolist()
    def take(self, i):
        self.sizes.pop(i)
        return self.ids.pop(i)
    def put(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        self.sizes.insert(i, SIZE[d]); self.ids.insert(i, d)

def new_pools():
    """Une réserve par (type, sous droits) : ('n', 0) = datasets n libres, ('n', 1) = sous droits..."""
    return {key: Pool(g) for key, g in datasets.groupby(["data_type", "is_copyrighted"])}

def fill(pool, lower, upper):
    """Prend dans la réserve des datasets dont la somme tombe dans [lower, upper]. Rend la liste, ou None (rien n'est pris)."""
    taken, total = [], 0
    while total < lower:
        i = bisect.bisect_left(pool.sizes, lower - total)              # le plus petit qui termine
        if i < len(pool.sizes) and pool.sizes[i] <= upper - total:
            j = i
        else:                                                          # sinon le plus gros qui rentre
            j = bisect.bisect_right(pool.sizes, upper - total) - 1
            if j < 0:
                break
        total += pool.sizes[j]
        taken.append(pool.take(j))
    if lower <= total <= upper:
        return taken
    for d in taken:
        pool.put(d)
    return None

# Exemple : le premier besoin du modèle 1
pools = new_pools()
t, lo, hi = NEEDS[1][0]
got = fill(pools[(t, 0)], lo, hi)
print(f"besoin {t} dans [{lo}, {hi}] -> datasets libres {got}, tailles {[SIZE[d] for d in got]}, somme {sum(SIZE[d] for d in got)}")

besoin n dans [79024, 81915] -> datasets libres [480, 7012, 8167, 9607, 6829, 1978, 9790, 3730], tailles [9994, 9969, 9959, 9953, 9948, 9931, 9916, 9374], somme 79044


## 3. Ce que rapporte et ce que coûte chaque modèle
Même tableau que dans le notebook 08, avec une colonne de plus : la **densité** `valeur / énergie`. C'est le critère classique du sac à dos.

In [6]:
m = models.copy()
m["valeur_si_libre"] = m.value
m["valeur_si_malus"] = m.value // 2
m["densité"] = m.value / m.energy_cost
m["part_du_plafond"] = m.energy_cost / energy_cap
display(m.sort_values("densité", ascending=False).head(8))

print(f"énergie totale de tous les modèles : {models.energy_cost.sum() / energy_cap:.2f} × le plafond")
print("modèles à énergie quasi nulle (≤ 10) :", (models.energy_cost <= 10).sum())
print(f"part du plafond pour un modèle médian : {m.part_du_plafond.median():.4f}  -> environ {1 / m.part_du_plafond.median():.0f} modèles médians")

,value,energy_cost,n_req,valeur_si_libre,valeur_si_malus,densité,part_du_plafond
model_id,,,,,,,
2359,240,1,2,240,120,240.0,4.775545e-15
621,235,1,1,235,117,235.0,4.775545e-15
2017,235,1,3,235,117,235.0,4.775545e-15
1000,224,1,2,224,112,224.0,4.775545e-15
1787,215,1,1,215,107,215.0,4.775545e-15
35,214,1,4,214,107,214.0,4.775545e-15
1542,195,1,1,195,97,195.0,4.775545e-15
2694,195,1,1,195,97,195.0,4.775545e-15


énergie totale de tous les modèles : 7.28 × le plafond
modèles à énergie quasi nulle (≤ 10) : 157
part du plafond pour un modèle médian : 0.0024  -> environ 415 modèles médians


**Lecture**
- L'énergie de tous les modèles fait **7,3 fois le plafond** : il faut choisir environ un modèle sur sept en énergie.
- 157 modèles ne coûtent presque rien : on les prendra en premier.
- **Deux ressources se disputent** : l'énergie (combien de modèles) et les données libres (combien de modèles à pleine valeur). Le plan idéal doit arbitrer entre les deux.

## 4. Avant de chercher : jusqu'où peut-on monter ?
Ici, la culture « solution exacte » redevient utile. On ne sait pas résoudre exactement le problème complet. En revanche, on sait résoudre exactement une **version relâchée** du problème, où l'on autorise à prendre « une fraction de modèle ». Sa solution est forcément **au moins aussi bonne** que n'importe quel vrai plan : c'est une **borne supérieure**. Elle permet de dire « mon plan est à X % de l'optimum au plus ».

### 4a. Borne 1 : sac à dos fractionnaire (énergie seule)
On ignore les données, on trie par densité et on remplit le plafond, en coupant le dernier modèle.

In [7]:
v_max = models.value.where(~models.index.isin(impossible_models), models.value // 2)   # un modèle impossible vaut au mieux la moitié
order = (v_max / models.energy_cost).sort_values(ascending=False).index
cum = models.energy_cost[order].cumsum()
k = int((cum <= energy_cap).sum())
borne_1 = v_max[order[:k]].sum() + (energy_cap - cum.iloc[k - 1]) / models.energy_cost[order[k]] * v_max[order[k]]
print(f"borne 1 (énergie seule) : {borne_1:,.0f}  avec {k} modèles entiers")

borne 1 (énergie seule) : 338,924  avec 862 modèles entiers


### 4b. Borne 2 : programme linéaire (énergie + données libres)
On ajoute les données. Pour chaque modèle, deux variables entre 0 et 1 :
- `x` : entraîné **à pleine valeur**, donc uniquement avec des données libres ;
- `y` : entraîné **à demi-valeur**, avec n'importe quelles données (ou une source).

$$\max \sum_i v_i\,x_i + \lfloor v_i/2 \rfloor\,y_i \quad\text{sous}\quad \sum_i e_i (x_i+y_i) \le E_{\max},\quad \sum_i L_{i,t}\,x_i \le \text{libre}_t \;\;\forall t,\quad x_i + y_i \le 1$$

`L_{i,t}` est le besoin minimal du modèle `i` en type `t`. Un modèle à pleine valeur consomme au moins `L_{i,t}` de données libres. C'est un programme linéaire : `scipy.optimize.linprog` le résout **exactement** en une fraction de seconde.

In [8]:
TYPES = list("ntic")
n = len(models)
v = models.value.to_numpy(float)
e = models.energy_cost.to_numpy(float)
half = (models.value // 2).to_numpy(float)
possible = ~models.index.isin(impossible_models)
L = valid_req.pivot_table(index="model_id", columns="data_type", values="lower", aggfunc="sum").reindex(models.index).fillna(0)[TYPES].to_numpy()

c = -np.concatenate([v * possible, half])                 # linprog minimise : on prend l'opposé
rows, rhs = [np.concatenate([e, e])], [energy_cap]          # énergie
for j, t in enumerate(TYPES):                               # données libres, par type
    rows.append(np.concatenate([L[:, j], np.zeros(n)]))
    rhs.append(datasets.query("data_type == @t and is_copyrighted == 0")["size"].sum())
A = vstack([csr_matrix(np.array(rows)), hstack([identity(n), identity(n)])])
b = np.concatenate([rhs, np.ones(n)])

t0 = time.time()
lp = linprog(c, A_ub=A, b_ub=b, bounds=(0, 1), method="highs")
borne_2 = -lp.fun
x_lp, y_lp = lp.x[:n], lp.x[n:]
print(f"borne 2 (énergie + données libres) : {borne_2:,.0f}   résolue en {time.time() - t0:.2f} s")
print(f"solution du PL : {(x_lp > 0.999).sum()} modèles pleins, {(y_lp > 0.999).sum()} à demi-valeur, "
      f"{((x_lp > 1e-6) & (x_lp < 0.999)).sum() + ((y_lp > 1e-6) & (y_lp < 0.999)).sum()} fractionnaires")

borne 2 (énergie + données libres) : 272,634   résolue en 0.07 s
solution du PL : 535 modèles pleins, 359 à demi-valeur, 7 fractionnaires


**Lecture**
- La borne 1 (339 k) est très optimiste : elle suppose que tous les modèles sont à pleine valeur.
- La borne 2 (272,6 k) est bien plus serrée. **Le vrai goulot n'est pas seulement l'énergie, ce sont les données libres** : la solution du PL met environ 40 % des modèles à demi-valeur.
- La solution du PL est presque entière : seuls quelques modèles sont « fractionnaires ». C'est un indice précieux pour la suite.
- On ignore ici les sources. C'est sans risque : une source ne rapporte rien et ne peut servir qu'un modèle déjà à demi-valeur, ce que `y` couvre déjà.

## 5. Les plans possibles
Chaque plan est une **règle de priorité** : on parcourt les modèles dans un ordre donné et, pour chacun, on essaie de remplir tous ses besoins (fonction `fill`). Si les données ou l'énergie manquent, on passe au suivant. Toute la différence entre les plans tient dans **l'ordre** et dans **le choix des données** (libres ou sous droits).

Une seule fonction sert à tous les plans :

In [9]:
def try_model(model, pools, statuses):
    """Remplit tous les besoins du modèle. statuses = statuts à essayer dans l'ordre (0 = libre, 1 = sous droits)."""
    got = []
    for t, lo, hi in NEEDS[model]:
        res = None
        for s in statuses:
            res = fill(pools[(t, s)], lo, hi)
            if res is not None:
                break
        if res is None:                       # un besoin échoue : on rend tout ce qu'on avait pris
            for d in got:
                pools[(TYPE[d], COPY[d])].put(d)
            return None
        got += res
    return got

def greedy(candidates):
    """candidates : liste de (modèle, statuts) par ordre de priorité. Un modèle n'est pris qu'une fois."""
    pools, energy, chosen, data_mappings = new_pools(), 0, set(), []
    for model, statuses in candidates:
        if model in chosen or model in impossible_models:
            continue
        if energy + models.energy_cost[model] > energy_cap:
            continue
        got = try_model(model, pools, statuses)
        if got is not None:
            chosen.add(model)
            energy += models.energy_cost[model]
            data_mappings += [[int(d), int(model)] for d in got]
    return {"dataMappings": data_mappings, "modelMappings": []}

results = {}
def evaluate(name, plan, seconds):
    score, valid, message = getSolutionScore(json.dumps(plan), DATASET_TXT, show_details=False)
    results[name] = dict(score=score, valide=valid, modèles=len({mm for _, mm in plan["dataMappings"]}),
                         part_borne=f"{score / borne_2:.2%}", secondes=round(seconds, 2))
    print(f"Plan {name} : score = {score:7d} | valide = {valid} {message} | {score / borne_2:.2%} de la borne")

### Plan A : les plus précieux d'abord
L'idée la plus naïve : trier par valeur décroissante, données libres de préférence.

In [10]:
t0 = time.time()
plan_a = greedy([(mm, (0, 1)) for mm in models.sort_values("value", ascending=False).index])
evaluate("A", plan_a, time.time() - t0)

Plan A : score =  198527 | valide = True  | 72.82% de la borne


### Plan B : la meilleure densité d'abord
Le réflexe du sac à dos : trier par `valeur / énergie`. Les modèles à énergie quasi nulle passent en tête.

In [11]:
density = (models.value / models.energy_cost).sort_values(ascending=False)
t0 = time.time()
plan_b = greedy([(mm, (0, 1)) for mm in density.index])
evaluate("B", plan_b, time.time() - t0)

Plan B : score =  256911 | valide = True  | 94.23% de la borne


### Plan C : un modèle, deux options
Le plan B gaspille des données libres. Dès qu'un seul dataset sous droits entre dans un modèle, **toute** sa valeur est divisée par deux, et les datasets libres qu'il a reçus ne servent plus à rien.

On donne donc à chaque modèle **deux options**, rangées dans la même liste de priorité :
- *pleine valeur* : densité `v / e`, données **libres uniquement** ;
- *demi-valeur* : densité `⌊v/2⌋ / e`, données **sous droits d'abord**, pour laisser les libres aux autres.

In [12]:
options = pd.concat([
    pd.DataFrame({"model": models.index, "density": models.value / models.energy_cost, "statuses": [(0,)] * n}),
    pd.DataFrame({"model": models.index, "density": (models.value // 2) / models.energy_cost, "statuses": [(1, 0)] * n}),
]).sort_values("density", ascending=False, kind="stable")
by_options = list(zip(options.model, options.statuses))

t0 = time.time()
plan_c = greedy(by_options)
evaluate("C", plan_c, time.time() - t0)

Plan C : score =  260035 | valide = True  | 95.38% de la borne


### Plan D : redistribuer les données libres
Le plan C choisit bien **quels** modèles entraîner (l'énergie est remplie), mais il distribue les données libres « au fil de l'eau ». Or les données libres doivent aller aux modèles qui en tirent le plus : passer à pleine valeur rapporte `v − ⌊v/2⌋`, pour un coût d'environ `Σ lower` en données libres.

On garde les modèles du plan C et on refait la distribution par **gain par unité de données libres** décroissant. Puis on complète avec la liste du plan C s'il reste de l'énergie.

In [13]:
upgrade = ((models.value - models.value // 2) / valid_req.groupby("model_id").lower.sum()).sort_values(ascending=False)

def models_of(plan):
    return {mm for _, mm in plan["dataMappings"]}

t0 = time.time()
chosen_c = [mm for mm in upgrade.index if mm in models_of(plan_c)]
plan_d = greedy([(mm, (0,)) for mm in chosen_c] + [(mm, (1, 0)) for mm in chosen_c] + by_options)
evaluate("D", plan_d, time.time() - t0)

Plan D : score =  269777 | valide = True  | 98.95% de la borne


### Plan E : suivre la solution du programme linéaire
Le PL de la partie 4 a fait l'arbitrage énergie / données libres **exactement**, mais en autorisant des fractions. Comme sa solution est presque entière, on la suit :
1. d'abord les modèles que le PL met à pleine valeur (`x > 0`), avec des données libres ;
2. puis ceux qu'il met à demi-valeur (`y > 0`), avec des données sous droits ;
3. les modèles du PL qui n'ont pas trouvé de données libres, à demi-valeur ;
4. enfin la liste du plan C, pour combler ce qui reste.

C'est la technique classique de **relaxation puis arrondi** : l'exact guide, l'heuristique répare.

In [14]:
lp_sol = pd.DataFrame({"model": models.index, "x": x_lp, "y": y_lp, "upgrade": models.index.map(upgrade)})
full_first = lp_sol[lp_sol.x > 1e-6].sort_values(["x", "upgrade"], ascending=False).model.tolist()
half_next = lp_sol[lp_sol.y > 1e-6].sort_values("y", ascending=False).model.tolist()

t0 = time.time()
plan_e = greedy([(mm, (0,)) for mm in full_first] + [(mm, (1, 0)) for mm in half_next]
                + [(mm, (1, 0)) for mm in full_first] + by_options)
evaluate("E", plan_e, time.time() - t0)

Plan E : score =  271511 | valide = True  | 99.59% de la borne


### Et les sources ?
Dans le notebook 08, une source faisait gagner le plan. Ici, seuls les 58 modèles « impossibles » en ont absolument besoin. Combien peuvent-ils rapporter au maximum ?

In [15]:
gain_max_sources = (models.loc[sorted(impossible_models), "value"] // 2).sum()
print(f"gain maximal si TOUS les modèles impossibles étaient servis gratuitement : {gain_max_sources:,}")
print(f"soit {gain_max_sources / results['E']['score']:.2%} du score du plan E")
print(f"écart restant entre le plan E et la borne : {borne_2 - results['E']['score']:,.0f}")

gain maximal si TOUS les modèles impossibles étaient servis gratuitement : 1,113
soit 0.41% du score du plan E
écart restant entre le plan E et la borne : 1,123


Même si **toutes** les sources étaient gratuites, le gain maximal serait de 1 113 points, soit 0,4 %. En pratique, il est presque nul : une source doit être un modèle à un seul besoin, et elle **coûte** son énergie (ou sa propre valeur, si on la retire du plan). Les modèles impossibles valent peu (38 en médiane, contre 500 pour un modèle moyen), et beaucoup ont plusieurs besoins impossibles. Un essai complet (non reproduit ici) ne fait gagner que 4 points.

Remarque : ce gain maximal est presque égal à l'écart entre le plan E et la borne. La borne compte en effet ces modèles à demi-valeur sans payer leurs sources. **Sur cette instance, les sources ne sont pas un levier.**

## 6. Comparaison des plans

In [16]:
summary = pd.DataFrame(results).T
summary.loc["borne 2 (PL)", "score"] = round(borne_2)
summary

,score,valide,modèles,part_borne,secondes
A,198527,True,362,72.82%,0.07
B,256911,True,844,94.23%,0.08
C,260035,True,846,95.38%,0.15
D,269777,True,846,98.95%,0.53
E,271511,True,876,99.59%,0.11
borne 2 (PL),272634,NaN,NaN,NaN,NaN


## 7. Pourquoi le plan E gagne
- **A → B (+29 %)** : trier par densité plutôt que par valeur. C'est la leçon du sac à dos : un modèle très précieux mais très gourmand bloque la place de plusieurs autres.
- **B → C (+1 %)** : ne pas mélanger données libres et sous droits dans un même modèle. Un seul dataset sous droits suffit à diviser la valeur par deux.
- **C → D (+3,7 %)** : les données libres sont la ressource rare. On les donne aux modèles qui en tirent le plus de valeur par unité.
- **D → E (+0,6 %)** : le programme linéaire fait l'arbitrage énergie / données libres de façon globale, alors que les règles gloutonnes le font modèle par modèle.
- Le plan E est à **moins de 0,5 % de la borne**. Il reste au plus 1 123 points à gagner, et sans doute beaucoup moins : la borne n'est pas forcément atteignable.

In [17]:
score, valid, message = getSolutionScore(json.dumps(plan_e), DATASET_TXT, show_details=True)
print("score =", score, "| valide =", valid)


Total value: 271511
Energy used: 209399476339804 / 209400189201537 (100.00%)
Number of used models: 876 / 3000 (29.20%)
  Number of devalued models: 339 (38.70%)
  Number of models used as data source: 0 (0.00%)
Number of used datasets: 5113 / 10000 (51.13%)
score = 271511 | valide = True


## 8. Écrire le fichier de solution

In [18]:
out_dir = ROOT_DIR / "solutions"; out_dir.mkdir(exist_ok=True)
out = out_dir / "2_medium_main.json"
out.write_text(json.dumps(plan_e))
print("écrit :", out)

écrit : /home/claude/proj/solutions/2_medium_main.json


## À retenir
1. **Quand on ne peut plus tout essayer, on choisit un ordre.** Un algorithme glouton n'est rien d'autre qu'une règle de priorité. Toute l'intelligence est dans cette règle.
2. **Il faut identifier la vraie ressource rare.** Ici, l'énergie décide combien de modèles on entraîne, mais ce sont les **données libres** qui décident combien sont à pleine valeur.
3. **Un malus « tout ou rien » change la stratégie.** Puisqu'un seul dataset sous droits divise la valeur par deux, on ne mélange jamais : un modèle est soit 100 % libre, soit nourri de données sous droits.
4. **L'exact et l'heuristique se complètent.** Le programme linéaire ne donne pas de solution utilisable, mais il donne une **borne** (on sait où on en est) et une **direction** (l'ordre du plan E).
5. **Les sources étaient décisives sur 5 modèles, elles sont négligeables sur 3 000.** Une astuce qui marche sur un petit exemple ne se transpose pas forcément.